# APIM ❤️ Microsoft Foundry

## Multi-Provider APIM Gateway lab (with Terraform)

Playground to demonstrate how to place **Azure AI Foundry** behind **Azure API Management** for high availability, priority-based load balancing, and unified access to **both OpenAI and Anthropic Claude** models using **Terraform** as infrastructure-as-code.

Key capabilities showcased:
- **Priority-based backend pool**: gpt-4.1-mini on Sweden Central (priority 1) with automatic failover to East US 2 (priority 2)
- **Circuit breaker**: each individual backend trips on HTTP 429 for 60 s, respecting `Retry-After` headers
- **Retry policy**: APIM transparently retries to the next available backend — callers never see a 429
- **Anthropic Claude 3.5 Haiku**: exposed through the same APIM gateway under a separate `/anthropic` path via Foundry Marketplace
- **Built-in LLM logging**: `azure-openai-emit-token-metric` for OpenAI; `emit-metric` for Anthropic — both flow to Application Insights
- **Terraform IaC**: all resources defined with `azurerm` + `azapi` providers

### TOC
- [0️⃣ Initialize notebook variables](#0)
- [1️⃣ Verify the Azure CLI and the connected Azure subscription](#1)
- [2️⃣ Create deployment using Terraform](#2)
- [3️⃣ Get the deployment outputs](#3)
- [🧪 Test the OpenAI API (priority failover)](#test-openai)
- [🧪 Test the Anthropic API](#test-anthropic)
- [🧪 Multi-provider LangChain agent](#test-langchain)
- [🗑️ Clean up resources](#clean)

### Prerequisites
- [Python 3.12 or later version](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) installed with the [Jupyter notebook extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) enabled
- [Python environment](https://code.visualstudio.com/docs/python/environments#_creating-environments) with the [requirements.txt](../../../requirements.txt)
- [An Azure Subscription](https://azure.microsoft.com/free/) with Contributor + RBAC Administrator or Owner roles
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed
- [Terraform CLI](https://developer.hashicorp.com/terraform/install) installed

▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`...

<a id='0'></a>
### 0️⃣ Initialize notebook variables

- Resources will be suffixed by a unique random string.
- `openai_model_capacity` is intentionally low to trigger throttling and demonstrate priority failover.
- Anthropic Claude requires Marketplace access in swedencentral — check availability in [Azure AI Studio model catalog](https://ai.azure.com/explore/models).

In [ ]:
import os, sys, json
sys.path.insert(1, '../../shared')
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}"
resource_group_location = "swedencentral"

apim_sku = "BasicV2_1"

openai_backends_config = """
{
  foundry-swc = {
    name     = \"foundry-openai-swc\"
    location = \"swedencentral\"
    priority = 1
    weight   = 100
  }
  foundry-eus2 = {
    name     = \"foundry-openai-eus2\"
    location = \"eastus2\"
    priority = 2
    weight   = 100
  }
}
"""

openai_model_deployment_name = "gpt-4.1-mini"
openai_model_name            = "gpt-4.1-mini"
openai_model_version         = "2025-04-14"
openai_model_capacity        = 20
openai_api_version           = "2024-10-21"

anthropic_foundry_name           = "foundry-anthropic-swc"
anthropic_foundry_location       = "swedencentral"
anthropic_model_deployment_name  = "claude-3-5-haiku"
anthropic_model_name             = "claude-3-5-haiku-20241022"
anthropic_model_capacity         = 1

utils.print_ok('Notebook initialized')

<a id='1'></a>
### 1️⃣ Verify the Azure CLI, Terraform CLI and the connected Azure subscription

The following commands ensure the Azure CLI is signed in to your Azure subscription and the Terraform CLI is installed.

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']
    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

In [ ]:
output = utils.run("terraform version", "Terraform CLI is installed", "Terraform CLI not found - install from https://developer.hashicorp.com/terraform/install")

<a id='2'></a>
### 2️⃣ Create deployment using Terraform

Writes `terraform.tfvars` from notebook variables and runs `terraform init` + `terraform apply`.
APIM BasicV2 provisioning typically takes 15–20 minutes.

In [ ]:
terraform_variables = (
    f'resource_group_name          = "{resource_group_name}"\n'
    f'resource_group_location      = "{resource_group_location}"\n'
    f'apim_sku                     = "{apim_sku}"\n'
    f'openai_model_deployment_name = "{openai_model_deployment_name}"\n'
    f'openai_model_name            = "{openai_model_name}"\n'
    f'openai_model_version         = "{openai_model_version}"\n'
    f'openai_model_capacity        = {openai_model_capacity}\n'
    f'openai_api_version           = "{openai_api_version}"\n'
    f'anthropic_foundry_name           = "{anthropic_foundry_name}"\n'
    f'anthropic_foundry_location       = "{anthropic_foundry_location}"\n'
    f'anthropic_model_deployment_name  = "{anthropic_model_deployment_name}"\n'
    f'anthropic_model_name             = "{anthropic_model_name}"\n'
    f'anthropic_model_capacity         = {anthropic_model_capacity}\n'
    f'openai_backends_config       = {openai_backends_config}\n'
)

with open('terraform.tfvars', 'w') as f:
    f.write(terraform_variables)

utils.print_ok('terraform.tfvars written')

In [ ]:
os.environ['ARM_SUBSCRIPTION_ID'] = subscription_id

output = utils.run('terraform init', 'Terraform initialization succeeded', 'Terraform initialization failed')
output = utils.run('terraform apply -auto-approve -var-file=terraform.tfvars', 'Terraform deployment succeeded', 'Terraform deployment failed')

<a id='3'></a>
### 3️⃣ Get the deployment outputs

In [ ]:
apim_resource_gateway_url = utils.run('terraform output apim_resource_gateway_url').json_data
utils.print_info(f'APIM Gateway URL: {apim_resource_gateway_url}')

apim_openai_subscription_key = utils.run('terraform output apim_openai_subscription_key').json_data
utils.print_info(f'OpenAI Subscription Key: ****{apim_openai_subscription_key[-4:]}')

apim_anthropic_subscription_key = utils.run('terraform output apim_anthropic_subscription_key').json_data
utils.print_info(f'Anthropic Subscription Key: ****{apim_anthropic_subscription_key[-4:]}')

log_analytics_workspace_id = utils.run('terraform output log_analytics_workspace_id').json_data
utils.print_info(f'Log Analytics Workspace ID: {log_analytics_workspace_id}')

<a id='test-openai'></a>
### 🧪 Test the OpenAI API — priority failover

Send multiple requests to the OpenAI API. Sweden Central (priority 1) serves first. When throttled, APIM's circuit breaker trips and the retry policy transparently fails over to East US 2 (priority 2). Watch the `x-ms-region` header change.

Tip: Use the [tracing tool](../../tools/tracing.ipynb) to observe backend selection in detail.

In [ ]:
import requests, time

runs = 30
sleep_time_ms = 50
url = f"{apim_resource_gateway_url}/openai/deployments/{openai_model_deployment_name}/chat/completions?api-version={openai_api_version}"
messages = {'messages': [{'role': 'system', 'content': 'You are a helpful assistant. Reply in one sentence.'}, {'role': 'user', 'content': 'Which Azure region are you running in?'}]}

session = requests.Session()
session.headers.update({'api-key': apim_openai_subscription_key})

try:
    for i in range(runs):
        print(f'\n▶️  Run {i+1}/{runs}:')
        start = time.time()
        response = session.post(url, json=messages)
        elapsed = time.time() - start
        utils.print_response_code(response)
        print(f'⌚ {elapsed:.2f}s')
        region = response.headers.get('x-ms-region', 'unknown')
        print(f'x-ms-region: \x1b[1;32m{region}\x1b[0m')
        if response.status_code == 200:
            data = response.json()
            usage = data.get('usage', {})
            print(f"Tokens — prompt: {usage.get('prompt_tokens','?')}, completion: {usage.get('completion_tokens','?')}, total: {usage.get('total_tokens','?')}")
            print(f"💬 {data['choices'][0]['message']['content']}")
        else:
            print(f'Response: {response.text}')
        time.sleep(sleep_time_ms / 1000)
finally:
    session.close()

<a id='test-anthropic'></a>
### 🧪 Test the Anthropic API

Send a request to **Claude 3.5 Haiku** via the same APIM gateway. APIM handles managed-identity authentication to Foundry and tracks token usage in Application Insights.

In [ ]:
import requests

url = f"{apim_resource_gateway_url}/anthropic/{anthropic_model_deployment_name}/chat/completions"
payload = {'model': anthropic_model_deployment_name, 'messages': [{'role': 'user', 'content': 'Tell me one interesting fact about Sweden in one sentence.'}], 'max_tokens': 200}
headers = {'api-key': apim_anthropic_subscription_key, 'Content-Type': 'application/json'}

response = requests.post(url, json=payload, headers=headers)
utils.print_response_code(response)

if response.status_code == 200:
    data = response.json()
    usage = data.get('usage', {})
    print(f"Tokens — input: {usage.get('input_tokens','?')}, output: {usage.get('output_tokens','?')}")
    content = data.get('choices', [{}])[0].get('message', {}).get('content', '')
    print(f'💬 {content}')
else:
    print(f'Error: {response.text}')

<a id='test-langchain'></a>
### 🧪 Multi-provider LangChain agent

Build a [LangChain ReAct agent](https://python.langchain.com/docs/concepts/agents/) that uses **both providers** — `AzureChatOpenAI` for GPT-4.1-mini and `ChatAnthropic` for Claude 3.5 Haiku — all routed through the single APIM gateway URL.

Both `langchain-openai` and `langchain-anthropic` support custom `base_url`/`azure_endpoint` parameters, making APIM fully transparent to the SDKs.

In [ ]:
%pip install langchain langchain-openai langchain-anthropic --quiet

In [ ]:
from langchain_openai import AzureChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate

# AzureChatOpenAI — routed through APIM /openai path
llm_openai = AzureChatOpenAI(
    azure_endpoint=apim_resource_gateway_url,
    azure_deployment=openai_model_deployment_name,
    api_key=apim_openai_subscription_key,
    api_version=openai_api_version,
    temperature=0,
)

# ChatAnthropic — routed through APIM /anthropic path
llm_anthropic = ChatAnthropic(
    base_url=f"{apim_resource_gateway_url}/anthropic",
    api_key=apim_anthropic_subscription_key,
    model=anthropic_model_deployment_name,
    temperature=0,
)

@tool
def ask_openai(question: str) -> str:
    """Ask the OpenAI GPT-4.1-mini model via APIM. Input: a question string."""
    return llm_openai.invoke(question).content

@tool
def ask_anthropic(question: str) -> str:
    """Ask the Anthropic Claude 3.5 Haiku model via APIM. Input: a question string."""
    return llm_anthropic.invoke(question).content

tools = [ask_openai, ask_anthropic]

react_prompt = PromptTemplate.from_template(
    'You are a helpful orchestrator that can query multiple AI providers.\n\n'
    'Tools:\n{tools}\n\n'
    'Format:\n'
    'Question: the input question\n'
    'Thought: what to do\n'
    'Action: one of [{tool_names}]\n'
    'Action Input: input to the action\n'
    'Observation: result\n'
    '...\n'
    'Thought: I now know the final answer\n'
    'Final Answer: the final answer\n\n'
    'Question: {input}\n'
    'Thought: {agent_scratchpad}'
)

agent = create_react_agent(llm_openai, tools, react_prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=5, handle_parsing_errors=True)

result = agent_executor.invoke({
    'input': 'Ask both GPT-4.1-mini and Claude 3.5 Haiku: "What is Azure API Management?" Then write a one-paragraph summary combining both answers.'
})

print('\n' + '='*60)
print('Final Answer:')
print(result['output'])

<a id='clean'></a>
### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.
Use the [clean-up-resources notebook](clean-up-resources.ipynb) for that.